# TUM-VIE DEVO-Style Recording Transfer

Train on one TUM-VIE recording and evaluate on another to stress generalization.

Models compared:
- ConvLIFSNN (vision-only)
- IMUConditionedVisualSNN (vision + IMU)

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import optax
from tonic import transforms
from tonic.datasets import TUMVIE

import spyx.models as fm

SEED = 123
TRAIN_REC = "mocap-6dof"
TEST_REC = "running-easy"
SPATIAL_FACTOR = 0.25
N_TIME_BINS = 24
TRAIN_SAMPLES = 96
TEST_SAMPLES = 48
BATCH_SIZE = 8
EPOCHS = 3

data_root = Path("../data").resolve()
data_root.mkdir(parents=True, exist_ok=True)
sensor = TUMVIE.sensor_size
sensor_small = (int(sensor[0] * SPATIAL_FACTOR), int(sensor[1] * SPATIAL_FACTOR), sensor[2])

tfm = transforms.Compose([
    transforms.Downsample(spatial_factor=SPATIAL_FACTOR),
    transforms.ToFrame(sensor_size=sensor_small, n_time_bins=N_TIME_BINS),
])

In [ ]:
def quat_to_euler_xyz(qx, qy, qz, qw):
    t0 = 2.0 * (qw * qx + qy * qz)
    t1 = 1.0 - 2.0 * (qx * qx + qy * qy)
    rx = np.arctan2(t0, t1)
    t2 = 2.0 * (qw * qy - qz * qx)
    t2 = np.clip(t2, -1.0, 1.0)
    ry = np.arcsin(t2)
    t3 = 2.0 * (qw * qz + qx * qy)
    t4 = 1.0 - 2.0 * (qy * qy + qz * qz)
    rz = np.arctan2(t3, t4)
    return np.array([rx, ry, rz], dtype=np.float32)


def load_recording(recording, limit):
    ds = TUMVIE(save_to=str(data_root), recording=recording, transform=tfm)
    xs, us, ys = [], [], []
    n = min(limit, len(ds))
    for i in range(n):
        d, t = ds[i]
        left = np.asarray(d["events_left"], dtype=np.float32)
        right = np.asarray(d["events_right"], dtype=np.float32)
        x = np.concatenate([left, right], axis=1)

        imu = np.asarray(d["imu"], dtype=np.float32)
        imu = imu.mean(axis=0) if imu.ndim == 2 and imu.shape[0] > 0 else np.zeros((6,), dtype=np.float32)

        mocap = np.asarray(t["mocap"], dtype=np.float32)
        mocap = mocap[-1] if mocap.ndim == 2 else mocap
        trans = mocap[:3]
        rot = quat_to_euler_xyz(*mocap[3:7])
        y = np.concatenate([trans, rot], axis=0).astype(np.float32)

        xs.append(x)
        us.append(imu)
        ys.append(y)

    return jnp.asarray(np.stack(xs, axis=0)), jnp.asarray(np.stack(us, axis=0)), jnp.asarray(np.stack(ys, axis=0))


Xtr, Utr, Ytr = load_recording(TRAIN_REC, TRAIN_SAMPLES)
Xte, Ute, Yte = load_recording(TEST_REC, TEST_SAMPLES)
print("train:", Xtr.shape, Utr.shape, Ytr.shape)
print("test :", Xte.shape, Ute.shape, Yte.shape)

In [ ]:
def make_batch(x, u, y, bs, key):
    idx = jax.random.choice(key, x.shape[0], shape=(bs,), replace=False)
    return x[idx], u[idx], y[idx]


def baseline_forward(x):
    cfg = fm.ConvConfig(
        T=int(x.shape[1]),
        in_channels=int(x.shape[2]),
        hidden_channels=(32, 64),
        output_dim=6,
        dt=1.0,
    )
    return fm.ConvLIFSNN(cfg)(x)


def fusion_forward(x, imu):
    cfg = fm.IMUConditionedConfig(
        vision_cfg=fm.ConvConfig(
            T=int(x.shape[1]),
            in_channels=int(x.shape[2]),
            hidden_channels=(32, 64),
            output_dim=6,
            dt=1.0,
        ),
        imu_dim=int(imu.shape[-1]),
        imu_hidden=128,
    )
    return fm.IMUConditionedVisualSNN(cfg)(x, imu)


def fit_and_eval(forward_fn, use_imu=False):
    if use_imu:
        transformed = hk.without_apply_rng(hk.transform(lambda x, u: forward_fn(x, u)))
        params = transformed.init(jax.random.PRNGKey(SEED), Xtr[:BATCH_SIZE], Utr[:BATCH_SIZE])
    else:
        transformed = hk.without_apply_rng(hk.transform(lambda x: forward_fn(x)))
        params = transformed.init(jax.random.PRNGKey(SEED), Xtr[:BATCH_SIZE])

    opt = optax.adam(1e-3)
    opt_state = opt.init(params)

    @jax.jit
    def train_step(params, opt_state, xb, ub, yb):
        def loss_fn(p):
            pred = transformed.apply(p, xb, ub) if use_imu else transformed.apply(p, xb)
            mse = jnp.mean((pred - yb) ** 2)
            return mse, pred

        (loss, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, opt_state = opt.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        mae = jnp.mean(jnp.abs(pred - yb))
        return params, opt_state, loss, mae

    @jax.jit
    def evaluate(params, x, u, y):
        pred = transformed.apply(params, x, u) if use_imu else transformed.apply(params, x)
        mse = jnp.mean((pred - y) ** 2)
        mae = jnp.mean(jnp.abs(pred - y))
        return mse, mae

    key = jax.random.PRNGKey(SEED + 7)
    for epoch in range(EPOCHS):
        key, sk = jax.random.split(key)
        xb, ub, yb = make_batch(Xtr, Utr, Ytr, BATCH_SIZE, sk)
        params, opt_state, tr_mse, tr_mae = train_step(params, opt_state, xb, ub, yb)
        print(f"epoch {epoch+1:02d} train mse={tr_mse:.4f} mae={tr_mae:.4f}")

    te_mse, te_mae = evaluate(params, Xte, Ute, Yte)
    return float(te_mse), float(te_mae)


baseline_mse, baseline_mae = fit_and_eval(baseline_forward, use_imu=False)
fusion_mse, fusion_mae = fit_and_eval(fusion_forward, use_imu=True)

print({
    "baseline_transfer": {"mse": baseline_mse, "mae": baseline_mae},
    "fusion_transfer": {"mse": fusion_mse, "mae": fusion_mae},
})